In [1]:
!wget https://raw.githubusercontent.com/SecondGlanceFromAlice/pyrcel/refs/heads/master/examples/basic_run.py

--2026-10-13 14:11:19--  https://raw.githubusercontent.com/SecondGlanceFromAlice/pyrcel/refs/heads/master/examples/basic_run.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6929 (6.8K) [text/plain]
Saving to: ‘basic_run.py.1’

basic_run.py.1      100%[===================>]   6.77K  --.-KB/s    in 0.007s  

2026-10-13 14:11:20 (947 KB/s) - ‘basic_run.py.1’ saved [6929/6929]



In [2]:
# pylint: disable=too-few-public-methods,missing-function-docstring

In [ ]:
import time

import numpy as np
import pyrcel as pm
import xarray as xr
from basic_run import _plot
from IPython.display import HTML, display

from ccnact import SI, parcel

In [ ]:
parameters = {
    "updraft_m_s": 1.0,
    "kappa": (0.54, 0.3),
    "mu_m": (5e-8, 1e-8),
    "sigma": (2.0, 1.6),
    "N_per_cc": (1000.0, 2000.0),
    "T0": 283.0,
    "S0": -0.02,
    "P0_hPa": 850.0,
    "n_bins": 50,
    "RH": 0.99,
}

In [5]:
accumulation = pm.AerosolSpecies(
    "accumulation",
    pm.Lognorm(
        parameters["mu_m"][0] / SI.um, parameters["sigma"][0], parameters["N_per_cc"][0]
    ),
    kappa=parameters["kappa"][0],
    bins=parameters["n_bins"],
)
aitken = pm.AerosolSpecies(
    "aitken",
    pm.Lognorm(
        parameters["mu_m"][1] / SI.um, parameters["sigma"][1], parameters["N_per_cc"][1]
    ),
    kappa=parameters["kappa"][1],
    bins=parameters["n_bins"],
)

model = pm.ParcelModel(
    [accumulation, aitken],
    V=parameters["updraft_m_s"],
    T0=parameters["T0"],
    S0=parameters["S0"],
    P0=parameters["P0_hPa"] * SI.hPa,
    console=True,
)
out = model.run(100.0, output_dt=1.0, terminate=False, progress=True)
type(out)

17.76%
19.88%
21.89%
24.03%
26.05%
28.17%
30.35%
32.39%
34.48%
36.59%
38.71%
40.84%
42.84%
44.90%
46.98%
49.10%
51.12%
53.23%
55.37%


[pyrcel]   finished in 8.83 s (includes one-time XLA compile)


57.47%
59.98%
62.09%
64.21%
66.75%
68.91%
72.19%
76.18%
81.07%
87.09%
94.48%
100.00%

  Trajectory (post-hoc sample)
     t [s]    z [m]    T [K]     S [%]   wc [g/kg]
  ------------------------------------------------
      0.00      0.0   283.00   -2.0000   1.054e-04
      4.00      4.0   282.96   -1.8107   1.154e-04
      8.00      8.0   282.92   -1.6214   1.280e-04
     13.00     13.0   282.87   -1.3848   1.482e-04
     17.00     17.0   282.83   -1.1956   1.694e-04
     21.00     21.0   282.80   -1.0065   1.973e-04
     26.00     26.0   282.75   -0.7704   2.479e-04
     30.00     30.0   282.71   -0.5820   3.102e-04
     34.00     34.0   282.67   -0.3946   4.104e-04
     39.00     39.0   282.62   -0.1632   6.598e-04
     43.00     43.0   282.58    0.0148   1.140e-03
     47.00     47.0   282.55    0.1713   2.477e-03
     52.00     52.0   282.51    0.2684   8.061e-03
     56.00     56.0   282.49    0.2437   1.658e-02
     60.00     60.0   282.47    0.1985   2.592e-02
     65.00     6

pyrcel.model_output.ModelOutput

In [6]:
_plot(out, model, fig_path := "pyrcel.svg")
display(HTML(f'<img src="{fig_path}">'))

Saved figure to pyrcel.svg


In [7]:
# print(ccnact.__version__)
# wall_time = time.perf_counter()
# cpu_time = time.process_time()

length_scale = 10 * SI.m

n_a, RH, r_w, time, act, T = parcel(
    n_bins=parameters["n_bins"],
    p=parameters["P0_hPa"] * SI.hPa,
    T=parameters["T0"] * SI.K,
    RH=parameters["RH"],
    dt=(dt := length_scale / parameters["updraft_m_s"]),
    nt=100 * SI.m / length_scale,
    w=parameters["updraft_m_s"],
    kappa=[parameters["kappa"][0]] * 2,  # TODO!
    meanr=parameters["mu_m"],
    gstdv=parameters["sigma"],
    n_vol=np.asarray(parameters["N_per_cc"]) / SI.cm**3,
)


class Output:
    """Mimicking pyrcel output"""

    aerosols = ()

    @staticmethod
    def to_xarray():
        return xr.Dataset(
            {
                "S": (("time",), (RH - 1) * 100),
                "t_smax": time[np.argmax(RH)],
                "height": (("time",), (parameters["updraft_m_s"] * time)),
                "T": (("temperature",), T),
            },
            coords={
                "time": ("time", time),
            },
        )


class Model:
    """mimicking pyrcel model"""

    @staticmethod
    def summary():
        return {
            "S_max": np.amax(RH) - 1,
            "per_species": {"accumulation", "aitken"},
        }


print(Output.to_xarray()["t_smax"])
_plot(Output, Model, fig_path := "ccnact.svg")
display(HTML(f'<img src="{fig_path}">'))

<xarray.DataArray 't_smax' ()> Size: 8B
array(25.26404038)
Saved figure to ccnact.svg


first run of the cell: ~22s
2nd run ~23s